# Derive Kvasir Runtime Summary

This notebook calculates Kvasir invocation runtimes from `data/exported/kvasir_stats.csv` using `runtime.started_at` and `runtime.finished_at`.

In [ ]:
from pathlib import Path

import pandas as pd


def find_package_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'data' / 'exported' / 'kvasir_stats.csv').exists():
            return path
    raise FileNotFoundError('Could not find data/exported/kvasir_stats.csv')


PACKAGE_ROOT = find_package_root(Path.cwd())
INPUT_CSV = PACKAGE_ROOT / 'data' / 'exported' / 'kvasir_stats.csv'

df = pd.read_csv(INPUT_CSV)
df.head()

## Calculate Runtime

Runtime is measured as `runtime.finished_at - runtime.started_at` and reported in seconds and minutes.

In [ ]:
required = ['runtime.started_at', 'runtime.finished_at']
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

runtime = df[required].copy()
runtime['started_at'] = pd.to_datetime(runtime['runtime.started_at'], utc=True, errors='coerce')
runtime['finished_at'] = pd.to_datetime(runtime['runtime.finished_at'], utc=True, errors='coerce')
runtime['runtime_seconds'] = (runtime['finished_at'] - runtime['started_at']).dt.total_seconds()
runtime['runtime_minutes'] = runtime['runtime_seconds'] / 60

valid_runtime = runtime['runtime_seconds'].dropna()
if valid_runtime.empty:
    raise ValueError('No valid runtime values could be derived')

median_runtime_seconds = valid_runtime.median()
median_runtime_minutes = median_runtime_seconds / 60

print(f'Kvasir invocations with runtime: {len(valid_runtime)}')
print(f'Median runtime: {median_runtime_seconds:.3f} seconds')
print(f'Median runtime: {median_runtime_minutes:.3f} minutes')